<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# MONAI Label Pathology App - QuPath Use Case Example

***The pathology bioimage analysis with MONAI Label***

In this notebook, we show the example of the MONAI Label Pathology use case. The nuclei segmentation is demonstrated. This tutorial uses QuPath as the client viewer, and we demonstrate the end-to-end workflow with MONAI Label + QuPath integration.

<img src="./images/monailabel_arch_qupath.png" width="1000"/>

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath1.png" width="700"/></p>

---

> **Note**: This notebook is an adaptation of [monailabel_pathology_nuclei_segmentation_QuPath.ipynb](../toolkit/tutorials/monai/monailabel/monailabel_pathology_nuclei_segmentation_QuPath.ipynb) 
notebook released with MONAI Toolkit. 
We have removed cells for setup and data download which we have already down for you in this lab. 

---

### Table of Contents
**[0.0 QuPath Notebook Setup](#0.0-QuPath-Notebook-Setup)** <br>
**[1.0 Open noVNC Preinstalled QuPath Program](#1.0-Open-noVNC-Preinstalled-QuPath-Program)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 Configure QuPath with the Plugin](#1.1-Configure-QuPath-with-the-Plugin) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.2 Start MONAI Label Server](#1.2-Start-MONAI-Label-Server) <br>
**[2.0 Nuclei Auto-Segmentation with QuPath](#2.0-Nuclei-Auto-Segmentation-with-QuPath)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.1 Start QuPath, MONAI Label Plugin, and Fetch Next Image](#2.1-Start-QuPath,-MONAI-Label-Plugin,-and-Fetch-Next-Image) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.2 Select ROI by Rectangle Tool](#2.2-Select-ROI-by-Rectangle-Tool) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.3 Run Nuclei Segmentation](#2.3-Run-Nuclei-Segmentation) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.4 Submit ROI Label for Training](#2.4-Submit-ROI-Label-for-Training) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.5 Run Interactive ROI-based Segmentation](#2.5-Run-Interactive-ROI-based-Segmentation) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.6 Run NuClick Model Inference by Single Click](#2.6-Run-NuClick-Model-Inference-by-Single-Click) <br>
**[3.0 Training Nuclei Segmentation](#3.0-Training-Nuclei-Segmentation)** <br>
**[4.0 Repeat the Interactive Labeling Process](#4.0-Repeat-the-Interactive-Labeling-Process)** <br>
**[5.0 QuPath Conclusion](#5.0-QuPath-Conclusion)** <br>

---
# 0.0 QuPath Notebook Setup  
let's import some libs used in this notebook.

In [ ]:
import ipywidgets
import threading
import os
HOST_NAME=os.getenv('HOST_NAME')
HOST_IP=os.getenv('HOST_IP')
DLI_DATA_DIR=os.getenv('DLI_DATA_DIR')
os.chdir(DLI_DATA_DIR)

if len(HOST_NAME)==0:
    from requests import get
    HOST_NAME=get('https://api.ipify.org').text
    HOST_IP=HOST_NAME
print(f" HOST_NAME is {HOST_NAME} with IP {HOST_IP}")


---
# 1.0 Open noVNC Preinstalled QuPath Program 
This lab already has a VNC instance running with preinstalled programs:
- 3d Slicer
- Qupath
- ImageJ

Run the cell below, then click the link to open noVNC for QuPath (same one as for Slicer).

<div class="alert alert-block alert-info">
Password:<b>gtc25</b>
</div>    

In [ ]:
url_text='<h1><center><a href="http://'+HOST_NAME+'/x11/" target="_blank">'
url_text=url_text+'<img src="https://biii.eu/sites/default/files/2017-09/qupath.png" width="100" height="50"/>'
url_text=url_text+'<br>Click to open Qupath <br> in a new window</a></center></h1>'
# print (url_text)
ipywidgets.HTML(url_text)

## 1.1 Configure QuPath with the Plugin
Once you are in the VNC web page you should:
1. Click on the first icon to launch QuPath
2. Go to edit--> preferences then add the MONAI Label plugin location of 
`/home.researcher/QuPath/v0.3`

![image](images/qupath_enable_extension.png) 


3. Restart QuPath by: Quit that application then launch it again

![test](images/qupath_setserver.png) 

4. Then write the address of your MONAI Label server.


## 1.2 Start MONAI Label Server

The command will start MONAI Label server with all available models.
To start with a specific model, set the model name in the **--conf models** argument.

Such as:

```bash
monailabel start_server --app <path to pathology app> --studies <path to downloaded datafolder> --conf models segmentation_nuclei
```
In the command, ```--app``` specifies the pathology app path, ```--studies``` specifies the dataset folder path.

In [ ]:
os.chdir(DLI_DATA_DIR)

# !monailabel start_server --app apps/pathology --studies datasets/ --conf models segmentation_nuclei,nuclick 
!monailabel start_server \
    --app apps/pathology \
    --studies datasets/ \
    --conf models segmentation_nuclei

---
# 2.0 Nuclei Auto-Segmentation with QuPath

Below is a step-by-step tutorial for nuclei segmentation using QuPath. The pretrained model is provided and loaded by MONAI Label server.


## 2.1 Start QuPath, MONAI Label Plugin, and Fetch Next Image

- On the menu bar, navigate and click MONAI Label. Click *Next Sample/Patch* to load data specified in the start_server command. If QuPath starts from the beginning, the button will automatically load the next whole slide image.

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath2.png" alt="drawing" width="800"/></p>

 - Users can also use the **Next Sample/patch** button to select a patch ROI automatically, by selecting **Next Patch**. （Uncheck the **Next Patch** box to load the next whole slide image.）

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath3.png" alt="drawing" width="800"/></p>

## 2.2 Select ROI by Rectangle Tool

First, let's show patch inference with the rectangle ROI tool.

- Users can automatically select the rectangle ROI by fetching "Next Patch", or use the QuPath rectangle tool to pick an ROI.

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath4.png" alt="drawing" width="800"/></p>

## 2.3 Run Nuclei Segmentation

- After selecting the rectangle ROI, users can navigate to the MONAI Label plugin and click **Annotations**.
Next, select models in the pop-up interface, and click **OK** to run inference. 

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath5.png" alt="drawing" width="800"/></p>

MONAI Label server will run auto-segmentation with the selected patch data and return the segmentation with polygons.

## 2.4 Submit ROI Label for Training

- In the next step, users can edit the predicted nuclei segmentation or use NuClick single click segmentation to add/modify the annotation. Once the annotation is done, users can navigate to the MONAI Label plugin and click **Submit Label** to save the ground truth.

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath11.png" alt="drawing" width="800"/></p>

- Then click **Yes** to save the ROI label:

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath12.png" alt="drawing" width="800"/></p>

## 2.5 Run Interactive ROI-based Segmentation

- Users can also use the interactive ROI tool to get auto-segmentation with one ROI draw:

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath7.png" alt="drawing" width="800"/></p>

- Draw any rectangle ROI, and MONAI Label server will return segmentation immediately:

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath8.png" alt="drawing" width="800"/></p>


## 2.6 Run NuClick Model Inference by Single Click

- MONAI Label server provides NuClick models where users can do instance segmentation with just a single click:

- Use the **Interactor** tool in the toolbar

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath9.png" alt="drawing" width="800"/></p>

- With click-and-inference, users can get nuclei segmentation without an ROI:

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath10.png" alt="drawing" width="800"/></p>



---
# 3.0 Training Nuclei Segmentation

Once users finish annotations on several patch images, MONAI Label provides a training option to fine-tune the model.

- Navigate to MONAI Label plugin and click **Training**

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath13.png" alt="drawing" width="800"/></p>

- In the training option pop-up interface, select **segmentation_nuclei**, select training customized training options. Uncheck **pretrained** to start training the model from scratch. Finally, click **OK** to start training in MONAI Label server.

<p align = "center"><img src="../toolkit/tutorials/monailabel/figures/monailabel_pathology_qupath/qupath14.png" alt="drawing" width="800"/></p>



---
# 4.0 Repeat the Interactive Labeling Process

Repeat the fetch data and interactive learning process until a satisfied model is achieved or the entire datasets/WSIs are annotated. 


---
# 5.0 QuPath Conclusion

In this notebook, you learned:
- The basic usage of MONAI Label with the pathology app
- An introduction to the nuclei segmentation task and several inference strategies including ROI, NuClick ROI, and single click
- How to submit label and training options to fine-tune the model or train from scratch

In the pathology use case, we highlight the interactive learning process when labeling a whole slide image or a dataset. MONAI Label achieves annotation actively and gradually improves the model and segmentation performance during labeling. 


<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">